# Circuits with knobs: parameters
**Quantum Computing · Week 7 (Variational quantum circuits) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qc/w7d1.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A **parameterized circuit** has knobs: gate angles left open as symbols, filled in later. One template,
a whole family of circuits. These knobs are what quantum algorithms and quantum ML models train.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace

theta, phi = Parameter("θ"), Parameter("φ")
knobs = QuantumCircuit(1)
knobs.ry(theta, 0)
knobs.rz(phi, 0)
print(knobs.draw("text"))
print("parameters:", knobs.parameters)

bound = knobs.assign_parameters({theta: np.pi / 2, phi: np.pi / 2})       # fill in the knobs
print("θ = π/2, φ = π/2 gives", Statevector(bound).draw("text"), " (that's |+i⟩, up to a global phase)")

## An ansatz: a template with layers
A common pattern (a "hardware-efficient ansatz"): a layer of $R_Y$ rotations, a chain of CNOTs between
neighbors, repeated, and a final rotation layer. $n$ qubits and $L$ repetitions give $n(L + 1)$ knobs.

In [ ]:
def ansatz(n, layers):
    angles = ParameterVector("θ", n * (layers + 1))
    qc = QuantumCircuit(n)
    k = 0
    for layer in range(layers + 1):
        for q in range(n):
            qc.ry(angles[k], q)
            k += 1
        if layer < layers:
            for q in range(n - 1):
                qc.cx(q, q + 1)
    return qc

template = ansatz(3, 2)
print("knobs:", template.num_parameters)
template.draw("mpl", style="clifford")

Qiskit has this pattern built in: `qiskit.circuit.library.real_amplitudes(3, reps=2)` makes the same
kind of circuit.

## Expressibility: where can the knobs take you?
Pick random knob settings and see which states come out. With $R_Y$ alone, one qubit's arrow can only
sweep the x–z circle; with $R_Y$ and $R_Z$ it reaches the whole sphere.

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6), layout="constrained")
for ax, use_rz, title in [(axes[0], False, "RY only: one circle"), (axes[1], True, "RY then RZ: the whole sphere")]:
    pts = []
    for _ in range(400):
        circuit = QuantumCircuit(1)
        circuit.ry(rng.uniform(0, np.pi), 0)
        if use_rz:
            circuit.rz(rng.uniform(-np.pi, np.pi), 0)
        rho = DensityMatrix(circuit)
        pts.append([np.real(np.trace(rho.data @ m)) for m in (np.array([[0, 1], [1, 0]]), np.array([[0, -1j], [1j, 0]]))])
    pts = np.array(pts)
    ax.scatter(pts[:, 0], pts[:, 1], s=8, color="#7b5cfa")
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="#b9a7ff"))
    ax.set_aspect("equal")
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_xlabel(r"$\langle X\rangle$")
    ax.set_ylabel(r"$\langle Y\rangle$")
    ax.set_title(title)
plt.show()

Seen from above (the x–y plane), $R_Y$ alone stays on a line: $\langle Y\rangle$ is always 0.

## Your turn: knobs for a Bell state
The two-qubit ansatz: $R_Y(a)$ and $R_Y(b)$, a CNOT, then $R_Y(c)$ and $R_Y(d)$. Find knob settings that make
$|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt2$, either by reasoning or with a search over multiples of 45°.
Then try the same search without the CNOT: what's the best fidelity you can get?

In [ ]:
bell = Statevector([1, 0, 0, 1]) / np.sqrt(2)

def two_qubit(a, b, c, d, with_cnot=True):
    qc = QuantumCircuit(2)
    qc.ry(a, 0)
    qc.ry(b, 1)
    if with_cnot:
        qc.cx(0, 1)
    qc.ry(c, 0)
    qc.ry(d, 1)
    return qc

def fidelity(qc):
    return abs(Statevector(qc).inner(bell)) ** 2

print(fidelity(two_qubit(0, 0, 0, 0)))     # TODO: find a, b, c, d with fidelity 1, then search without the CNOT

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
print("a = 90°, the rest 0:", round(fidelity(two_qubit(np.pi / 2, 0, 0, 0)), 3), "  (RY(90°) makes |+>, the CNOT copies it)")
steps = np.arange(-180, 181, 45) * np.pi / 180
for with_cnot in [True, False]:
    best = max(fidelity(two_qubit(a, b, c, d, with_cnot)) for a in steps for b in steps for c in steps for d in steps)
    print(f"best fidelity {'with' if with_cnot else 'without'} the CNOT: {best:.3f}")
print("Without a CNOT the circuit only makes product states, and no product state is more than 50% Bell.")